# bandcopy — バンドコピー支援（Colab版）

自分の**手持ちの音源**をアップロードすると、演奏しやすい難易度に落とした
**楽譜・タブ譜**と、**パート別の練習音源**を作ります。個人練習用です。

このColabからは、**2つの画面**が1つのURLで使えます。

| 画面 | できること |
|---|---|
| **みんな用**（トップ） | 音源をアップロード → バンド譜PDF・タブ譜PDF・パート別練習音源をダウンロード |
| **ドラム編集**（`/editor/`） | ドラムの打点をマス目で編集 → その場で音を鳴らして確認 → 譜面・MIDIで書き出し |

## 使い方（3ステップ）
1. メニューの **「ランタイム」→「ランタイムのタイプを変更」→ ハードウェアアクセラレータ = GPU** にすると速くなります（任意）。
2. 上から順に、各セルの左にある **▶（実行ボタン）** を押します。
3. 最後のセルに出る **公開URL** を開きます。ドラム編集はそのURLの末尾に `/editor/` を付けたページです。

> 初回は準備（インストール・モデルのダウンロード）で数分かかります。

### 1. 準備（コードの取得とインストール）

In [ ]:
# bandcopy のコードを取得
REPO_URL = "https://github.com/jasawsea/bandcopy.git"

import os
if not os.path.isdir("bandcopy"):
    !git clone -q $REPO_URL
%cd bandcopy

# 依存をインストール（ffmpeg は Colab に導入済み）
!pip install -q -r requirements.txt
print("準備できました。次のセルを実行してください。")

### 2. アプリを起動

実行すると **一時公開URL** が表示されます。そのURLをLINE等で共有すれば、
他の人はColabを触らずブラウザだけで使えます（URLはこのセッションが動いている間だけ有効）。

- `https://〜.gradio.live` … みんな用（楽譜・タブ譜・練習音源）
- `https://〜.gradio.live/editor/` … ドラム編集

**このセルを実行したあとも、Colabのタブは閉じないでください。**閉じるとURLが切れます。

In [ ]:
# みんな用とドラム編集を「1つのURL」で公開する
import secrets, threading, time
import httpx, uvicorn
from gradio.networking import setup_tunnel
from serve_all import build_app

PORT = 7860

# サーバをバックグラウンドで起動（/=みんな用・/editor/=ドラム編集）
_config = uvicorn.Config(build_app(), host="127.0.0.1", port=PORT, log_level="error")
_server = uvicorn.Server(_config)
threading.Thread(target=_server.run, daemon=True).start()

for _ in range(60):                      # 起動待ち
    try:
        httpx.get(f"http://127.0.0.1:{PORT}/editor/", timeout=2)
        break
    except Exception:
        time.sleep(0.5)

# ポートごと公開する（Gradioの共有トンネル）。/editor/ もこのURL配下で使える
public_url = setup_tunnel("127.0.0.1", PORT, secrets.token_urlsafe(32), None, None)

print("みんな用　　：", public_url)
print("ドラム編集　：", public_url + "/editor/")